<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Launch Sites Locations Analysis with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:
- **TASK 1:** Mark all launch sites on a map
- **TASK 2:** Mark the success/failed launches for each site on the map
- **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [1]:
!pip3 install folium
!pip3 install wget
!pip3 install pandas

In [2]:
import folium
import wget
import pandas as pd

In [3]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/DV0101EN-3-5-1-Generating-Maps-in-Python-py-v2.0.ipynb)


## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site. 


In [4]:
# Download and read the `spacex_launch_geo.csv`
spacex_csv_file = wget.download('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df=pd.read_csv(spacex_csv_file)

100% [................................................................................] 7710 / 7710

Now, you can take a look at what are the coordinates for each site.


In [5]:
spacex_df.head()

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


In [6]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df
# alternate code
# launch_sites_df = spacex_df[['Launch Site', 'Lat', 'Long']].drop_duplicates()

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [7]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example, 


In [8]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle. 


Now, let's add a circle for each launch site in data frame `launch_sites`


_TODO:_  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [9]:
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Historical and operational differences between LC-40 & SLC-40

LC-40 (Launch Complex 40): This was the original name of the complex used by the US Air Force during the 1960s to 2000s for Titan III and Titan IV rockets.

SLC-40 (Space Launch Complex 40): This is the current name of the facility after being leased by SpaceX in 2007, rebuilt and upgraded to launch the Falcon 9 rocket.

Functional Difference: LC-40 was demolished/retrofitted to become SLC-40, which is now a SpaceX "workhorse" for satellite launches (Starlink, cargo missions and Dragon crew).

Summary: SLC-40 is the modern and upgraded version of the old LC-40, operated by SpaceX for the Falcon 9.

In [10]:
## Cape Canaveral Launch Complex 40 coordinate
CCAFS_LC_Lat_coord = launch_sites_df.loc[launch_sites_df['Launch Site'] == "CCAFS LC-40", 'Lat'].values[0]
CCAFS_LC_Long_coord = launch_sites_df.loc[launch_sites_df['Launch Site'] == "CCAFS LC-40", 'Long'].values[0]
CCAFS_LC_coordinate = CCAFS_LC_Lat_coord, CCAFS_LC_Long_coord
CCAFS_LC_coordinate 

(np.float64(28.56230197), np.float64(-80.57735648))

In [11]:
## Cape Canaveral Space Launch Complex 40 coordinate
CCAFS_SLC_Lat_coord = launch_sites_df.loc[launch_sites_df['Launch Site'] == "CCAFS SLC-40", 'Lat'].values[0]
CCAFS_SLC_Long_coord = launch_sites_df.loc[launch_sites_df['Launch Site'] == "CCAFS SLC-40", 'Long'].values[0]
CCAFS_SLC_coordinate = CCAFS_SLC_Lat_coord, CCAFS_SLC_Long_coord
CCAFS_SLC_coordinate 

(np.float64(28.56319718), np.float64(-80.57682003))

In [12]:
## Kennedy Space Center Launch Complex 39A coordinate
KSC_Lat_coord = launch_sites_df.loc[launch_sites_df['Launch Site'] == "KSC LC-39A", 'Lat'].values[0]
KSC_Long_coord = launch_sites_df.loc[launch_sites_df['Launch Site'] == "KSC LC-39A", 'Long'].values[0]
KSC_coordinate = KSC_Lat_coord, KSC_Long_coord
KSC_coordinate 

(np.float64(28.57325457), np.float64(-80.64689529))

In [13]:
## Vandenberg Space Launch Complex 4 coordinate
VAFB_Lat_coord = launch_sites_df.loc[launch_sites_df['Launch Site'] == "VAFB SLC-4E", 'Lat'].values[0]
VAFB_Long_coord = launch_sites_df.loc[launch_sites_df['Launch Site'] == "VAFB SLC-4E", 'Long'].values[0]
VAFB_coordinate = VAFB_Lat_coord, VAFB_Long_coord
VAFB_coordinate 

(np.float64(34.63283416), np.float64(-120.6107455))

In [29]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

In [30]:
# For each launch site, add a Circle object based on its coordinate (Lat, Long) values. In addition, add Launch site name as a popup label

circle_LC = folium.Circle(CCAFS_LC_coordinate, radius=50, color='#d35400').add_child(folium.Popup('Cape Canaveral Launch Complex 40'))
marker_LC = folium.map.Marker(
    CCAFS_LC_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'CCAFS LC-40',
        )
    )

circle_SLC = folium.Circle(CCAFS_SLC_coordinate, radius=50, color='#d35400').add_child(folium.Popup('Cape Canaveral Space Launch Complex 40'))
marker_SLC = folium.map.Marker(
    CCAFS_SLC_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'CCAFS SLC-40',
        )
    )

circle_KSC = folium.Circle(KSC_coordinate, radius=50, color='#d35400').add_child(folium.Popup('Kennedy Space Center Launch Complex 39A '))
marker_KSC = folium.map.Marker(
    KSC_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'KSC LC-39A',
        )
    )

circle_VAFB = folium.Circle(VAFB_coordinate, radius=50, color='#d35400').add_child(folium.Popup('Vandenberg Space Launch Complex 4 coordinate'))
marker_VAFB = folium.map.Marker(VAFB_coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'VAFB SLC-4E', )  )

site_map.add_child(circle)
site_map.add_child(marker)

site_map.add_child(circle_LC)
site_map.add_child(marker_LC)

site_map.add_child(circle_SLC)
site_map.add_child(marker_SLC)

site_map.add_child(circle_KSC)
site_map.add_child(marker_KSC)

site_map.add_child(circle_VAFB)
site_map.add_child(marker_VAFB)

The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:
- Are all launch sites in proximity to the Equator line?
- Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


Launch centers are located at the latitudes closest to the equator possible within the American territory.

This is because rocket launch centers must seek locations near the equator to better utilize the momentum of the Earth's rotation. 

Launching near the equator saves fuel, increases payload, and maximizes the efficiency of the space mission.

I believe that Hawaii was not considered as a launch center location due to access logistics and the risk of seismicity.

# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [16]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records. 
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [17]:
marker_cluster = MarkerCluster()

_TODO:_ Create a new column in `launch_sites` dataframe called `marker_color` to store the marker colors based on the `class` value


In [18]:
# Apply a function to check the value of `class` column
# If class=1, marker_color value will be green
# If class=0, marker_color value will be red

# Function to assign color to launch outcome
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'
    
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


_TODO:_ For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [31]:
# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

# for each row in spacex_df data frame
# create a Marker object with its coordinate
# and customize the Marker's icon property to indicate if this launch was successed or failed, 
# e.g., icon=folium.Icon(color='white', icon_color=row['marker_color']

for index, record in spacex_df.iterrows():
    coordinate = [record['Lat'], record['Long']]
    
    marker = folium.Marker(
        location=coordinate,
        icon=folium.Icon(color='white', icon_color=record['marker_color'],
                         html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % record['Launch Site'],))
    
    marker_cluster.add_child(marker)

site_map


Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [20]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


You can calculate the distance between two points on the map based on their `Lat` and `Long` values using the following method:


In [21]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

_TODO:_ Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [22]:
# find coordinate of the closet coastline
# e.g.,: Lat: 28.56367  Lon: -80.57163
# (np.float64(28.56230197), np.float64(-80.57735648))
distance_coastline = calculate_distance(28.56230197, -80.57735648, 28.56367, -80.57163)

In [23]:
coordinate = [28.56367 ,-80.57163]

In [24]:
distance_coastline

0.5797581813109574

_TODO:_ After obtained its coordinate, create a `folium.Marker` to show the distance


In [25]:
# Create and add a folium.Marker on your selected closest coastline point on the map
# Display the distance between coastline point and launch site using the icon property 
# for example
distance_marker = folium.Marker(
    coordinate,
    icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),))

# site_map.add_child(distance_marker)

_TODO:_ Draw a `PolyLine` between a launch site to the selected coastline point


In [26]:
#Launch Site	Lat	Long
#	CCAFS LC-40	    28.562302	-80.577356
#	CCAFS SLC-40	28.563197	-80.576820
#	KSC LC-39A	    28.573255	-80.646895
#	VAFB SLC-4E	    34.632834	-120.610745

In [35]:
# Create a `folium.PolyLine` object using the coastline and road coordinates and launch site 

SLC_dist_coast = calculate_distance(28.563197, -80.576820, 28.56293, -80.57078)
SLC_dist_road = calculate_distance(28.563197, -80.576820, 28.56293, -80.56782)

 
coordinate = [28.56293, -80.57078]
SLC_dist_coast_marker = folium.Marker(
    coordinate,
    icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
        html='<div style="font-size: 36; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(SLC_dist_coast),))

coordinate = [28.56293, -80.56782]
SLC_dist_road_marker = folium.Marker(
    coordinate,
    icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
        html='<div style="font-size: 36; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(SLC_dist_road),))

points = [
    [28.56230197, -80.57735648], # Launch Site
    [28.562930197, -80.57078] # roadline
]

road_line=folium.PolyLine(locations=points, color="red", weight=5)
site_map.add_child(road_line)
site_map.add_child(SLC_dist_road_marker)

points = [
    [28.56230197, -80.57735648], # Launch Site
    [28.562930197, -80.56782] # coast line
]

coast_line=folium.PolyLine(locations=points, color="blue", weight=5)
site_map.add_child(coast_line)
site_map.add_child(SLC_dist_coast_marker)

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


_TODO:_ Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [ ]:
# Create a marker with distance to a closest city, railway, highway, etc.
# Draw a line between the marker to the launch site


In [ ]:
Vanderberg_distance_coastline = calculate_distance(34.63283416, -120.6107455, 34.637, -120.62445)
Vanderberg_distance_coastline
coordinate = [34.637, -120.6244]
Vanderberg_distance_coastline = folium.Marker(
    coordinate,
    icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(Vanderberg_distance_coastline),))

points = [
    [34.63283416, -120.6107455], # Launch Site
    [34.637, -120.6244] # Cost line
]

Vanderberg_coast_lines=folium.PolyLine(locations=points, color="red", weight=5)
site_map.add_child(Vanderberg_coast_lines)
site_map.add_child(Vanderberg_distance_coastline)

In [ ]:
Vanderberg_distance_railroad = calculate_distance(34.63283416, -120.6107455, 34.63673, -120.62356)
Vanderberg_distance_railroad

coordinate = [34.63673, -120.62356]
Vanderberg_distance_railroad = folium.Marker(
    coordinate,
    icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(Vanderberg_distance_railroad),))

points = [
    [34.63283416, -120.6107455], # Launch Site
    [34.63673, -120.62356] # Cost line
]

Vanderberg_coastroad_lines=folium.PolyLine(locations=points, color="red", weight=5)
site_map.add_child(Vanderberg_railroad_lines)
site_map.add_child(Vanderberg_distance_railroad)

In [ ]:
Vanderberg_distance_coastroad = calculate_distance(34.63283416, -120.6107455, 34.63564, -120.62356)
Vanderberg_distance_coastroad

coordinate = [34.63673, -120.62356]
Vanderberg_distance_coastroad = folium.Marker(
    coordinate,
    icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(Vanderberg_distance_coastroad),))

points = [
    [34.63283416, -120.6107455], # Launch Site
    [34.63564, -120.62356] # Cost line
]

Vanderberg_railroad_lines=folium.PolyLine(locations=points, color="red", weight=5)
site_map.add_child(Vanderberg_coastroad_lines)
site_map.add_child(Vanderberg_distance_coastroad)

After you plot distance lines to the proximities, you can answer the following questions easily:
- Are launch sites in close proximity to railways?
- Are launch sites in close proximity to highways?
- Are launch sites in close proximity to coastline?
- Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


The launch sites are located near railways and railroads, and are also close to the coast, approximately 1.3 km away.

However, it is observed that the launch sites maintain a certain distance from cities, probably to reduce the risk of a launch accident harming residents.

# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/)


### Other Contributors


Joseph Santarcangelo


## Change Log


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2021-05-26|1.0|Yan|Created the initial version|


Copyright © 2021 IBM Corporation. All rights reserved.
